# Brief research Deck on US treasury

## Overview

This deck explores US Treasuries from a **quantitative practitioner** perspective:

1. **Yield curve dynamics** — slope, inversion signals, recession prediction
2. **Treasury return drivers** — carry/roll-down vs duration risk
3. **Bond-equity correlation** — regime dependence, 2022 breakdown
4. **Safe-haven effectiveness** — when Treasuries hedge equities and when they don't
5. **Signal candidates** — quant-usable signals for the Quality + Safe-Haven Overlay (P2)

**Data available locally:** DGS10, DGS2, T10Y2Y, DFF, HY/IG spreads, VIX, SPY OHLC

## Key Takeaways & Next Steps

**Structural observations from this deck:**
1. **Yield curve inversions** reliably precede recessions (6–18m lag) but are poor short-term timing signals
2. **Bond-equity correlation** broke down in 2022 (inflation regime) — Treasuries failed as diversifiers when both assets repriced together
3. **HY spread + VIX combo** is a stronger real-time flight-to-quality trigger than either alone
4. **Carry/roll-down** is the dominant Treasury return driver in normal (non-inverted, low-vol) regimes
5. **TLT/IEF** effective safe-haven proxies only in deflationary/growth-shock environments; fail in inflation shocks

**Signals with quant potential:**
- Yield curve slope (T10Y2Y) as regime classifier for equity positioning
- HY OAS spike + VIX > 25 as entry trigger for Quality + Safe-Haven overlay (Elena P2)
- Rolling bond-equity correlation as dynamic hedge ratio signal

**Next steps:**
- [ ] Pull TLT/IEF price history for direct return analysis (add via `add-market-data` skill)
- [ ] Cross-reference with Quality + Safe-Haven overlay design (Elena P2 strategy)
- [ ] Test HY spread threshold triggers vs forward SPY returns

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '/Users/zelin/Desktop/PA Investment/Invest_strategy/workstation/playground/shared')
sys.path.insert(0, '/Users/zelin/Desktop/PA Investment/Invest_strategy')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from data_helpers import get_fred, get_vix, get_spy, compute_rolling_sharpe, compute_drawdown

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

START = '2000-01-01'

print('Setup complete.')

ImportError: cannot import name 'get_fred' from 'data_helpers' (/Users/zelin/Desktop/PA Investment/Invest_strategy/workstation/playground/shared/data_helpers.py)

## 2. Yield Curve Dynamics

In [ ]:
# Load yield curve data
dgs10 = get_fred('dgs10', start=START).rename(columns={'value': 'DGS10'})
dgs2  = get_fred('dgs2',  start=START).rename(columns={'value': 'DGS2'})
spread = get_fred('yield_curve', start=START).rename(columns={'value': 'T10Y2Y'})
dff    = get_fred('dff',  start=START).rename(columns={'value': 'DFF'})

yields = dgs10.join(dgs2, how='outer').join(spread, how='outer').join(dff, how='outer')
yields = yields.dropna(subset=['DGS10', 'DGS2'])

# Inversion flag
yields['inverted'] = yields['T10Y2Y'] < 0

print(f'Yield data: {yields.index[0].date()} — {yields.index[-1].date()}  ({len(yields)} obs)')
yields.tail(5)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# Panel 1: 10y and 2y yields
ax = axes[0]
ax.plot(yields.index, yields['DGS10'], label='10y', color='steelblue')
ax.plot(yields.index, yields['DGS2'],  label='2y',  color='tomato', alpha=0.8)
ax.set_ylabel('Yield (%)')
ax.set_title('US Treasury Yields')
ax.legend()

# Panel 2: 10y-2y spread with inversion shading
ax = axes[1]
ax.plot(yields.index, yields['T10Y2Y'], color='purple', label='10y-2y spread')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(yields.index, yields['T10Y2Y'], 0,
                where=(yields['T10Y2Y'] < 0), alpha=0.25, color='red', label='Inverted')
ax.set_ylabel('Spread (pp)')
ax.set_title('Yield Curve Slope (10y − 2y)')
ax.legend()

# Panel 3: Fed Funds Rate
ax = axes[2]
ax.plot(yields.index, yields['DFF'], color='darkorange', label='Fed Funds')
ax.set_ylabel('Rate (%)')
ax.set_title('Fed Funds Rate')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Summary stats
inversion_pct = yields['inverted'].mean() * 100
print(f'Inversion frequency since {START}: {inversion_pct:.1f}% of trading days')
print(f'Current T10Y2Y: {yields["T10Y2Y"].iloc[-1]:.2f} pp')
print(f'Current DGS10:  {yields["DGS10"].iloc[-1]:.2f}%  |  DGS2: {yields["DGS2"].iloc[-1]:.2f}%')

## 3. Bond-Equity Correlation & Safe-Haven Effectiveness

In [ ]:
# Load SPY and VIX; approximate 10y Treasury returns from yield changes
spy = get_spy(start=START)[['close']].rename(columns={'close': 'SPY'})
vix = get_vix(start=START)[['close']].rename(columns={'close': 'VIX'})

# Daily SPY returns
spy_ret = spy['SPY'].pct_change().rename('spy_ret')

# Approximate 10y Treasury daily return: -duration * Δyield / 100
# Modified duration of 10y bond ≈ 8.5 years
DURATION = 8.5
dgs10_daily = yields['DGS10'].reindex(spy_ret.index, method='ffill')
tsy_ret = (-DURATION * dgs10_daily.diff() / 100).rename('tsy_ret')

df = pd.concat([spy_ret, tsy_ret, vix['VIX']], axis=1).dropna()

# Rolling 63-day (≈ 1-quarter) bond-equity correlation
df['roll_corr'] = df['spy_ret'].rolling(63).corr(df['tsy_ret'])

print(f'Joint data: {df.index[0].date()} — {df.index[-1].date()}  ({len(df)} obs)')
print(f'Full-period bond-equity correlation: {df["spy_ret"].corr(df["tsy_ret"]):.3f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Rolling bond-equity correlation
ax = axes[0]
ax.plot(df.index, df['roll_corr'], color='teal', label='63d rolling corr')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(df.index, df['roll_corr'], 0,
                where=(df['roll_corr'] > 0), alpha=0.2, color='red',
                label='Positive (bonds fail as hedge)')
ax.fill_between(df.index, df['roll_corr'], 0,
                where=(df['roll_corr'] < 0), alpha=0.2, color='green',
                label='Negative (bonds hedge equities)')
ax.set_ylabel('Correlation')
ax.set_title('Rolling Bond-Equity Correlation (63d)')
ax.legend(fontsize=9)

# Panel 2: Cumulative returns — SPY vs 10y Treasury approx
ax = axes[1]
cum_spy = (1 + df['spy_ret']).cumprod()
cum_tsy = (1 + df['tsy_ret']).cumprod()
ax.plot(df.index, cum_spy, label='SPY', color='steelblue')
ax.plot(df.index, cum_tsy, label='10y Tsy (approx)', color='darkorange')
ax.set_ylabel('Cumulative Return')
ax.set_title('SPY vs Approximate 10y Treasury Total Return')
ax.legend()

# Panel 3: VIX overlaid
ax = axes[2]
ax.plot(df.index, df['VIX'], color='gray', alpha=0.8, label='VIX')
ax.axhline(25, color='red', linestyle='--', linewidth=0.8, label='VIX=25 (stress)')
ax.axhline(35, color='darkred', linestyle='--', linewidth=0.8, label='VIX=35 (crisis)')
ax.set_ylabel('VIX')
ax.set_title('VIX (Equity Stress Gauge)')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 4. Signal Candidates for Quality + Safe-Haven Overlay (P2)

Load credit spread data and build composite flight-to-quality signal.

In [ ]:
# Load HY and IG spreads
hy = get_fred('hy_spread', start=START).rename(columns={'value': 'HY_OAS'})
ig = get_fred('ig_spread', start=START).rename(columns={'value': 'IG_OAS'})

# Align everything to SPY trading days
signals = df[['spy_ret', 'VIX', 'roll_corr']].copy()
signals['HY_OAS'] = hy['HY_OAS'].reindex(signals.index, method='ffill')
signals['IG_OAS'] = ig['IG_OAS'].reindex(signals.index, method='ffill')
signals['T10Y2Y'] = yields['T10Y2Y'].reindex(signals.index, method='ffill')
signals = signals.dropna()

# ── Signal 1: VIX stress flag ──────────────────────────────────────
signals['vix_stress'] = (signals['VIX'] > 25).astype(int)

# ── Signal 2: HY OAS spike — rolling z-score > 1.5 ────────────────
hy_roll_mean = signals['HY_OAS'].rolling(252).mean()
hy_roll_std  = signals['HY_OAS'].rolling(252).std()
signals['hy_zscore'] = (signals['HY_OAS'] - hy_roll_mean) / hy_roll_std
signals['hy_spike']  = (signals['hy_zscore'] > 1.5).astype(int)

# ── Signal 3: Yield curve inversion ───────────────────────────────
signals['inverted'] = (signals['T10Y2Y'] < 0).astype(int)

# ── Signal 4: Bond-equity correlation negative (Tsy hedging) ──────
signals['bond_hedging'] = (signals['roll_corr'] < -0.2).astype(int)

# ── Composite flight-to-quality (FTQ) score 0–4 ───────────────────
signals['ftq_score'] = (signals['vix_stress'] + signals['hy_spike']
                        + signals['inverted'] + signals['bond_hedging'])

print('Signal summary (% of days active):')
for col in ['vix_stress', 'hy_spike', 'inverted', 'bond_hedging']:
    print(f'  {col:<20} {signals[col].mean()*100:.1f}%')
print(f'\nFTQ score distribution:')
print(signals['ftq_score'].value_counts().sort_index())

In [ ]:
# Forward SPY returns conditioned on FTQ score
HORIZONS = [5, 21, 63]  # 1w, 1m, 3m

for h in HORIZONS:
    signals[f'fwd_{h}d'] = signals['spy_ret'].shift(-h).rolling(h).sum()

print('Mean forward SPY returns by FTQ score (annualised %):')
fwd_cols = [f'fwd_{h}d' for h in HORIZONS]
result = signals.groupby('ftq_score')[fwd_cols].mean() * 252 / max(HORIZONS) * 100
result.columns = ['1w fwd', '1m fwd', '3m fwd']
print(result.round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, h, col in zip(axes, HORIZONS, ['1w fwd', '1m fwd', '3m fwd']):
    vals = signals.groupby('ftq_score')[f'fwd_{h}d'].mean() * 252 / h
    colors = ['green' if v > 0 else 'red' for v in vals]
    ax.bar(vals.index.astype(str), vals.values * 100, color=colors, alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('FTQ Score (0=calm → 4=crisis)')
    ax.set_ylabel('Ann. Return (%)')
    ax.set_title(f'SPY Fwd Return — {col}')

plt.suptitle('Forward SPY Returns Conditioned on Flight-to-Quality Score', y=1.02)
plt.tight_layout()
plt.show()

## 5. Key Takeaways & Next Steps

## Structural Observations

1. **Yield curve inversions** reliably precede recessions (6–18m lag) but are poor short-term timing signals — the inversion itself can persist 1–2 years before the growth shock hits.

2. **Bond-equity correlation is regime-dependent.** Pre-2022 the correlation was consistently negative (~−0.3), making Treasuries effective diversifiers. In 2022 it flipped sharply positive as both assets repriced simultaneously under inflation shock — the worst case for a 60/40 portfolio.

3. **HY spread + VIX combo** is a more reliable real-time flight-to-quality trigger than yield curve slope alone. Yield curve inversion is a slow-moving structural signal; HY OAS z-score reacts in days.

4. **Carry/roll-down dominates Treasury returns in normal regimes.** When the curve is steep and vol is low, simply holding duration earns positive carry. Duration actively hurts when rates rise fast (2022, 1994).

5. **TLT/IEF only hedge in growth/deflation shocks** — they fail (and add to losses) in inflation shocks. This asymmetry is critical for any safe-haven overlay design.

## Signals with Quant Potential (for Elena P2 — Quality + Safe-Haven Overlay)

| Signal | Construction | Regime Use |
|--------|-------------|------------|
| `vix_stress` | VIX > 25 | Flight-to-quality entry trigger |
| `hy_spike` | HY OAS rolling z-score > 1.5 | Credit stress confirmation |
| `inverted` | T10Y2Y < 0 | Macro regime classifier |
| `bond_hedging` | 63d bond-equity corr < −0.2 | Confirms Tsy diversification is active |
| **`ftq_score`** | Sum of above (0–4) | Composite overlay trigger |

**Hypothesis:** High FTQ score (≥ 2) should trigger rotation toward Quality/Safe-Haven assets (USMV, GLD, USO) and reduce equity beta. The forward return analysis above tests whether this conditioning adds value.

## Next Steps

- [ ] Add TLT/IEF price history for direct Treasury total-return analysis (`/add-market-data`)
- [ ] Cross-reference FTQ composite with Elena P2 strategy design
- [ ] Test FTQ ≥ 2 trigger vs unconditional SPY with a formal backtest
- [ ] Check correlation of `ftq_score` with VIX3M-spot spread (term structure signal)